In [ ]:
# Placeholder parameters: illustrative values only, not the ones used in live research.
SPIKE_MULT = 1.3
END_W = 0.5
LOW_BAND = 0.10
HIGH_BAND = 0.30
BACK_IV_RATIO = 0.9
VOV_SCALE = 10
TAKE_PROFIT_MULT = 2.0
PROFIT_CAP = 1.0
EXPIRY_WEEKS = 3

# VIX Spike Detection
X day rolling average < X * VIX

spike end on day when X * spike_high + X *  X day rolling average before spike

In [ ]:
import pandas as pd

# Load and prepare VIX data
vix_df = pd.read_csv("CBOE Volatility Index Historical Data 2021 to 2025.csv")
vix_df["Date"] = pd.to_datetime(vix_df["Date"])
vix_df["Price"] = pd.to_numeric(vix_df["Price"], errors="coerce")
vix_df = vix_df.sort_values("Date").reset_index(drop=True)

# Calculate X-day rolling average
vix_df["VIX_5d_avg"] = vix_df["Price"].rolling(window=5).mean()

# Define a spike: VIX > X × X-day avg
vix_df["Spike"] = vix_df["Price"] > SPIKE_MULT * vix_df["VIX_5d_avg"]

# Group consecutive spike days
spike_groups = []
current_group = []

for i in range(len(vix_df)):
    if vix_df.loc[i, "Spike"]:
        current_group.append(i)
    elif current_group:
        spike_groups.append(current_group)
        current_group = []

if current_group:
    spike_groups.append(current_group)

# Find spike end based on new definition
spike_events = []

for group in spike_groups:
    start_idx = group[0]
    end_idx = group[-1]

    # Spike prices and the pre-spike average
    spike_prices = vix_df.loc[group, "Price"].tolist()
    spike_high = max(spike_prices)

    # Get the day of the spike high
    spike_high_idx = group[spike_prices.index(spike_high)]
    spike_high_date = vix_df.loc[spike_high_idx, "Date"]

    pre_spike_avg = vix_df.loc[start_idx, "VIX_5d_avg"]

    if pd.isna(pre_spike_avg):
        continue  # Skip if not enough data for pre-spike avg

    # New Spike End Threshold: END_W * spike_high + (X - END_W) * pre_spike_avg
    spike_end_threshold = END_W * spike_high + (1 - END_W) * pre_spike_avg
    spike_end_date = None

    # Search for the first day after the spike where VIX drops below this threshold
    for i in range(end_idx + 1, len(vix_df)):
        if vix_df.loc[i, "Price"] < spike_end_threshold:
            spike_end_date = vix_df.loc[i, "Date"]
            break

    # Calculate post-spike X-day average (starting the day after the spike end)
    post_spike_5d_avg = None
    if spike_end_date:
        post_spike_idx = vix_df[vix_df["Date"] == spike_end_date].index[0] + 1
        if post_spike_idx + 4 < len(vix_df):
            post_spike_5d_avg = vix_df.loc[post_spike_idx:post_spike_idx + 4, "Price"].mean()

    # Calculate spike duration (trading days between start and end)
    if spike_end_date:
        spike_duration = pd.bdate_range(vix_df.loc[start_idx, "Date"], spike_end_date).size
    else:
        spike_duration = None  # In case no valid end date is found

    spike_events.append({
        "Start Date": vix_df.loc[start_idx, "Date"],
        "Spike End Date": spike_end_date,
        "Spike High": spike_high,
        "Day of Spike High": spike_high_date,
        "Pre-Spike 5D Avg": pre_spike_avg,
        "Spike End Threshold": spike_end_threshold,
        "Post-Spike 5D Avg": post_spike_5d_avg,
        "Spike Duration (Trading Days)": spike_duration,
        "Spike Prices": spike_prices
    })

# Create a result DataFrame
spike_result_df = pd.DataFrame(spike_events)
print(spike_result_df)


# Backtest
Buy puts every day of the VIX spike

Expiration date of put is the closest Friday X weeks from the buy date

Settle on expiration

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm

# Load and prepare VIX data
vix_df = pd.read_csv("CBOE Volatility Index Historical Data 2021 to 2025.csv")
vix_df["Date"] = pd.to_datetime(vix_df["Date"])
vix_df["Price"] = pd.to_numeric(vix_df["Price"], errors="coerce")
vix_df = vix_df.sort_values("Date").reset_index(drop=True)

# Calculate X-day rolling average
vix_df["VIX_5d_avg"] = vix_df["Price"].rolling(window=5).mean()

# Define a spike: VIX > X × X-day avg
vix_df["Spike"] = vix_df["Price"] > SPIKE_MULT * vix_df["VIX_5d_avg"]

# Calculate daily returns for VIX
vix_df["VIX_Return"] = vix_df["Price"].pct_change()

# Calculate X-day rolling standard deviation of returns (VoV)
vix_df["VoV_10d"] = vix_df["VIX_Return"].rolling(window=10).std()

# Group consecutive spike days
spike_groups = []
current_group = []

for i in range(len(vix_df)):
    if vix_df.loc[i, "Spike"]:
        current_group.append(i)
    elif current_group:
        spike_groups.append(current_group)
        current_group = []

if current_group:
    spike_groups.append(current_group)

# Black-Scholes formula for put option
def black_scholes_put(S, K, T, r, sigma):
    """ Black-Scholes formula for European put option price """
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    put_price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    return put_price

def black_scholes_call(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + sigma**2 / 2.0) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

# Function to get the closest Friday X weeks after the purchase date
def get_expiration_date(purchase_date):
    # Add X days to the purchase date
    expiration_date = purchase_date + pd.Timedelta(days=EXPIRY_WEEKS * 7)

    # If expiration is on a weekend, adjust to the Friday before the weekend
    if expiration_date.weekday() == 4:
        return expiration_date
    else:
        # Calculate the number of days to subtract or add to get the closest Friday
        days_to_friday = (4 - expiration_date.weekday()) % 7
        closest_friday = expiration_date + pd.Timedelta(days=days_to_friday)
        return closest_friday

    return expiration_date

# Simulate the returns from buying X put every day during the spike and cash settling on expiration
put_simulation_results = []

for group in spike_groups:
    start_idx = group[0]
    end_idx = group[-1]

    # Strike prices: VIX closing prices for each day of the spike
    strike_prices = np.floor(vix_df.loc[group, "Price"].tolist())

    # Assume a constant risk-free rate (r) and expiration time (T)
    r = 0.01  # Risk-free rate
    T = EXPIRY_WEEKS / 52  # X weeks to expiration in years

    # Calculate the spike end threshold: average of the highest VIX price and pre-spike X-day average
    spike_prices = vix_df.loc[group, "Price"].tolist()
    spike_high = max(spike_prices)
    pre_spike_avg = vix_df.loc[start_idx, "VIX_5d_avg"]

    if pd.isna(pre_spike_avg):
        continue  # Skip if not enough data for pre-spike avg

    # Spike end threshold = average of pre-spike X-day avg and spike high
    spike_end_threshold = END_W * spike_high + (1 - END_W) * pre_spike_avg

    # Simulate buying X put each day during the spike and settling at post-spike expiration
    total_return = 0
    for i, strike_price in enumerate(strike_prices):
        # Use VoV as implied volatility for Black-Scholes model
        implied_volatility = vix_df.loc[group[i], "VoV_10d"] * VOV_SCALE
        strike_price = strike_price

        # Calculate the Black-Scholes put price for the current day
        put_price = black_scholes_put(S=vix_df.loc[group[i], "Price"], K=strike_price, T=T, r=r, sigma=implied_volatility)

        # Get the expiration date (X days from the buy date)
        purchase_date = vix_df.loc[group[i], "Date"]
        expiration_date = get_expiration_date(purchase_date)

        # Ensure the expiration date exists in the dataset
        expiration_row = vix_df[vix_df["Date"] == expiration_date]

        if not expiration_row.empty:
            expiration_price = expiration_row["Price"].values[0]

            # Cash settlement at expiration: Option's payoff = max(strike - expiration_price, X)
            option_payoff = max(strike_price - expiration_price, 0) - put_price
            # print(strike_price, expiration_price, put_price)
            total_return += option_payoff * 1  # X contract per day

    # Store results for each spike
    put_simulation_results.append({
        "Spike Start Date": vix_df.loc[start_idx, "Date"],
        "End Date (Post-Spike Expiration)": expiration_date,
        # "Strike Prices": strike_prices,
        "Contract Prices" : put_price,
        "Total Return (Simulated)": total_return,
        "Num of trades": len(group)
    })

# Create a result DataFrame
put_simulation_df = pd.DataFrame(put_simulation_results)
print(put_simulation_df)

trading_return = put_simulation_df["Total Return (Simulated)"]

# --- Sharpe Ratio ---
sharpe_ratio = trading_return.mean() / trading_return.std() * np.sqrt(35)  # X periods/year
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")

# --- CAGR ---
initial_value = 1
cumulative_return = (1 + trading_return).cumprod()
final_value = cumulative_return.iloc[-1]
years = len(trading_return) / 35
cagr = (final_value / initial_value) ** (1 / years) - 1
print(f"CAGR: {cagr:.2%}")

# --- Max Drawdown (compounded) ---
rolling_max = cumulative_return.cummax()
drawdown = cumulative_return / rolling_max - 1
max_drawdown = drawdown.min()
print(f"Max Drawdown: {max_drawdown:.2%}")

# --- Win Rate ---
win_rate = (trading_return > 0).mean()
print(f"Win Rate: {win_rate:.2%}")

# Backtest sell calendar spreads

VIX spike between X% and X%

sell calls expiring next week at current VIX

buy calls expiring next X weeks at current VIX

close both positions on the day the sell call expires

assume X weeks IV = X week IV * X

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm

# Load your real VIX data
vix_df = pd.read_csv("CBOE Volatility Index Historical Data 2021 to 2025.csv")
vix_df["Date"] = pd.to_datetime(vix_df["Date"])
vix_df["Price"] = pd.to_numeric(vix_df["Price"], errors="coerce")
vix_df = vix_df.sort_values("Date").reset_index(drop=True)

# Clean and calculate
vix_df["VIX_5d_avg"] = vix_df["Price"].rolling(window=5).mean()
vix_df["VIX_Return"] = vix_df["Change %"].str.replace('%', '').astype(float) / 100
vix_df["VoV_10d"] = vix_df["VIX_Return"].rolling(window=10).std()

# Black-Scholes pricing for a call
def black_scholes_call(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + sigma**2 / 2.0) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def get_expiration_date(purchase_date, d, dte):
    # Add X days to the purchase date
    expiration_date = purchase_date + pd.Timedelta(days=dte)

    # If expiration is on a weekend, adjust to the Wednesday before the weekend
    if expiration_date.weekday() == 2:
        return expiration_date
    else:
        # Calculate the number of days to subtract or add to get the closest Wednesday
        days_to_wed = (2 - expiration_date.weekday()) % 7
        closest_wed = expiration_date + pd.Timedelta(days=days_to_wed)
        return closest_wed

# Calendar Spread Backtest
calendar_spread_results = []

for i in range(5, len(vix_df) - 25):  # leave room for XD horizon
    vix_return = vix_df.loc[i, "VIX_Return"]
    current_price = vix_df.loc[i, "Price"]
    current_date = vix_df.loc[i, "Date"]
    seven_day_dte = get_expiration_date(current_date, 2, 7)
    thirty_day_dte = get_expiration_date(current_date, 2, 21)

    # Entry condition: VIX spikes X%–X%
    if LOW_BAND < vix_return <= HIGH_BAND:
        strike = np.floor(current_price)
        front_iv = vix_df.loc[i, "VoV_10d"] * VOV_SCALE
        back_iv = front_iv * BACK_IV_RATIO  # assume steep term structure

        # Prices for XD and XD calls

        front_call_price = black_scholes_call(current_price, strike, (seven_day_dte - current_date).days/365, 0.001, front_iv)
        back_call_price = black_scholes_call(current_price, strike, (thirty_day_dte - current_date).days/365, 0.001, back_iv)
        net_cost = back_call_price - front_call_price

        # Exit X days later, value of back-month call with XD remaining
        if i + 11 < len(vix_df):
            future_price = vix_df.loc[vix_df[vix_df["Date"] == seven_day_dte].index, "Price"]
            remaining_call_value = black_scholes_call(future_price, strike, (thirty_day_dte - seven_day_dte).days/365, 0.001, back_iv)
            profit = remaining_call_value - net_cost

            calendar_spread_results.append({
                "Trade Date": current_date,
                "VIX Return (%)": vix_return * 100,
                "Strike": strike,
                "Front Call Price": front_call_price,
                "Back Call Price": back_call_price,
                "Net Cost": net_cost,
                "Future VIX (Day+7)": future_price,
                "Remaining Value of Long Call": remaining_call_value,
                "Profit": profit
            })

# Convert to DataFrame
calendar_df = pd.DataFrame(calendar_spread_results)
print(calendar_df)

# Optional: performance metrics
if not calendar_df.empty:
    profits = calendar_df["Profit"].apply(float)
    mean_return = profits.mean()
    std_return = profits.std()
    sharpe_ratio = mean_return / std_return
    print(f"\nTotal Trades: {len(profits)}")
    print(f"Total Profit: {profits.sum():.2f}")
    print(f"Mean Profit per Trade: {mean_return:.2f}")
    print(f"Sharpe Ratio: {sharpe_ratio:.2f}")


# Backtest buy put
Buy puts every day of the VIX spike

Buy put with strike price at closing price rounded down

Sell put with strike price at VIX X day average before spike rounded down

Expiration date of put is the closest Friday X weeks from the buy date

Settle on expiration

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm

# Load and prepare VIX data
vix_df = pd.read_csv("CBOE Volatility Index Historical Data 2021 to 2025.csv")
vix_df["Date"] = pd.to_datetime(vix_df["Date"])
vix_df["Price"] = pd.to_numeric(vix_df["Price"], errors="coerce")
vix_df = vix_df.sort_values("Date").reset_index(drop=True)

# Calculate X-day rolling average
vix_df["VIX_5d_avg"] = vix_df["Price"].rolling(window=5).mean()

# Calculate daily returns for VIX
vix_df["VIX_Return"] = vix_df["Change %"].str.replace('%', '').astype(float) / 100

# Calculate X-day rolling standard deviation of returns (VoV)
vix_df["VoV_10d"] = vix_df["VIX_Return"].rolling(window=10).std()

# Function to calculate the average VIX for a given period
def calculate_avg_vix(start_idx, end_idx):
    return vix_df.loc[start_idx:end_idx, "Price"].mean()

# Function to get the closest Friday X weeks after the purchase date
def get_expiration_date(purchase_date, d=4, dte=14):
    # Add X days to the purchase date
    expiration_date = purchase_date + pd.Timedelta(days=dte)

    # If expiration is on a weekend, adjust to the Friday before the weekend
    if expiration_date.weekday() == d:
        return expiration_date
    else:
        # Calculate the number of days to subtract or add to get the closest Friday
        days_to_friday = (d - expiration_date.weekday()) % 7
        closest_friday = expiration_date + pd.Timedelta(days=days_to_friday)
        return closest_friday

# Black-Scholes formula for put option
def black_scholes_put(S, K, T, r, sigma):
    """ Black-Scholes formula for European put option price """
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    put_price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    return put_price

def black_scholes_call(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + sigma**2 / 2.0) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

# Simulate the returns from buying X put every day during the spike and cash settling on expiration
put_simulation_results = []

# Group consecutive spike days
spike_groups = []
current_group = []

for i in range(1, len(vix_df)):
    # Detect when VIX spikes > X%
    thresh = HIGH_BAND
    if len(current_group) > 0 and vix_df.loc[i, "VIX_Return"] > 0:
        current_group.append(i)
    elif vix_df.loc[i, "VIX_Return"] > thresh:  # If VIX spikes by more than X%
        current_group.append(i)
    elif len(current_group) > 0:
        # Add completed group to the spike_groups list
        spike_groups.append(current_group)
        current_group = []

# Add the last group if any
if len(current_group) > 0:
    spike_groups.append(current_group)

merged_groups = []
merged_group = spike_groups[0]  # Start with the first group

for i in range(1, len(spike_groups)):
    # Get the end date of the previous group (last date in the previous group)
    prev_group_end_date = vix_df.loc[merged_group[-1], "Date"]

    # Get the start date of the current group (first date in the current group)
    curr_group_start_date = vix_df.loc[spike_groups[i][0], "Date"]

    # Check if the current group starts within X days of the previous group’s end date
    if (curr_group_start_date - prev_group_end_date).days <= 14:
        # If yes, merge the groups by appending the current group’s indexes to the merged group
        merged_group.extend(spike_groups[i])
    else:
        # If no, add the current merged group to the merged_groups list and start a new group
        merged_groups.append(merged_group)
        merged_group = spike_groups[i]  # Start a new group

# Append the last group
merged_groups.append(merged_group)

spike_groups = merged_groups

total_cost = []

# Simulate the options strategies
for group in spike_groups:
    start_idx = group[0]
    end_idx = group[-1]

    # Get the pre-spike X-day average
    pre_spike_avg = vix_df.loc[start_idx, "VIX_5d_avg"]

    # Ensure we have enough data for the pre-spike average
    if pd.isna(pre_spike_avg):
        continue


    # Define the pre-spike period (X days before the spike)
    pre_spike_start = start_idx - 5
    pre_spike_end = start_idx - 1

    # Define the post-spike period (X days after the spike ends)
    post_spike_start = end_idx + 1
    post_spike_end = end_idx + 5

    # Calculate the VIX averages before and after the spike
    if pre_spike_start >= 0 and post_spike_end < len(vix_df):
        # Calculate the VIX averages
        pre_spike_avg = np.mean(vix_df.loc[pre_spike_start:pre_spike_end, "Price"])
        post_spike_avg = np.mean(vix_df.loc[post_spike_start:post_spike_end, "Price"])

        # Calculate the percentage change in VIX before and after the spike
        vix_change = (post_spike_avg - pre_spike_avg) / pre_spike_avg * 100

        # Simulate buying and selling puts for each day during the spike
        total_return = 0
        for j in range(len(group)):
            long_strike = np.floor(vix_df.loc[group[j], "Price"])  # Long put strike price (ATM)
            short_strike = np.floor(vix_df.loc[start_idx, "VIX_5d_avg"])  # Short put strike price (pre-spike X-day avg)

            # Use VoV as implied volatility for Black-Scholes model
            implied_volatility = vix_df.loc[group[j], "VoV_10d"] * VOV_SCALE  # Scaling VoV to match IV for long put
            implied_volatility_short = vix_df.loc[group[j], "VoV_10d"] * VOV_SCALE  # Scaling VoV for short put

            # Calculate the Black-Scholes put price for the current day (long put)
            long_put_price = black_scholes_put(S=vix_df.loc[group[j], "Price"], K=long_strike, T=EXPIRY_WEEKS / 52, r=0.01, sigma=implied_volatility)

            # Calculate the Black-Scholes put price for the pre-spike X-day average (short put)
            short_put_price = black_scholes_put(S=vix_df.loc[group[j], "Price"], K=short_strike, T=EXPIRY_WEEKS / 52, r=0.01, sigma=implied_volatility_short)

            total_cost.append(long_put_price-short_put_price)

            # Get the expiration date (X days from the buy date)
            purchase_date = vix_df.loc[group[j], "Date"]
            expiration_date = get_expiration_date(purchase_date, 4)

            # Ensure the expiration date exists in the dataset
            expiration_row = vix_df[vix_df["Date"] == expiration_date]

            if not expiration_row.empty:
                expiration_price = expiration_row["Price"].values[0]

                # Cash settlement at expiration: Option's payoff = max(strike - expiration_price, X)
                long_option_payoff = max(long_strike - expiration_price, 0)  # Long put payoff
                short_option_payoff = max(short_strike - expiration_price, 0)  # Short put payoff

                # Net payoff: long put payoff minus short put payoff
                net_payoff = long_option_payoff - short_option_payoff - long_put_price + short_put_price
                if vix_df.loc[group[j], "VIX_Return"] > HIGH_BAND:
                    total_return += net_payoff * 2
                else:
                    total_return += net_payoff * 1

        # Store results for each spike, including long and short contract prices
        put_simulation_results.append({
            "Start Date": vix_df.loc[start_idx, "Date"],
            "End Date": vix_df.loc[end_idx, "Date"],
            "Expiration Close Price": expiration_price,
            "Long Strike Prices": long_strike,
            "Long Contract Price": long_put_price,  # Long put contract price
            "Short Strike Price (Pre-Spike 5D Avg)": short_strike,
            "Short Contract Price": short_put_price,  # Short put contract price
            "Spread Difference": long_strike - short_strike,
            "Total Return (Simulated)": total_return,
            "Spike Duration (Days)": len(group),
            "Pre-Spike VIX Average": pre_spike_avg,
            "Post-Spike VIX Average": post_spike_avg,
        })

# Calendar Spread Backtest
calendar_spread_results = []

for i in range(5, len(vix_df) - 25):  # leave room for XD horizon
    vix_return = vix_df.loc[i, "VIX_Return"]
    current_price = vix_df.loc[i, "Price"]
    current_date = vix_df.loc[i, "Date"]
    seven_day_dte = get_expiration_date(current_date, 2, 7)
    thirty_day_dte = get_expiration_date(current_date, 2, 21)

    # Entry condition: VIX spikes X%–X%
    if LOW_BAND < vix_return <= HIGH_BAND:
        strike = np.floor(current_price)
        long_strike = np.floor(current_price)
        front_iv = vix_df.loc[i, "VoV_10d"] * VOV_SCALE
        back_iv = front_iv * BACK_IV_RATIO  # assume steep term structure

        # Prices for XD and XD calls

        front_call_price = black_scholes_call(current_price, strike, (seven_day_dte - current_date).days/365, 0.001, front_iv)
        back_call_price = black_scholes_call(current_price, long_strike, (thirty_day_dte - current_date).days/365, 0.001, back_iv)
        net_cost = back_call_price - front_call_price

        total_cost.append(net_cost)

        # Exit X days later, value of back-month call with XD remaining
        if i + 11 < len(vix_df):
            mask = vix_df["Date"] == seven_day_dte
            if not mask.any():
                raise ValueError(f"No price found for {seven_day_dte}")
            future_price = vix_df.loc[mask, "Price"].iloc[0]
            # print(future_price)
            remaining_call_value = black_scholes_call(future_price, long_strike, (thirty_day_dte - seven_day_dte).days/365, 0.001, back_iv)
            profit = remaining_call_value - net_cost - max(future_price - strike, 0) + front_call_price
            # profit = -max(future_price - strike, X) + front_call_price

            calendar_spread_results.append({
                "Trade Date": current_date,
                "Strike": strike,
                "Front Call Price": front_call_price,
                "Back Call Price": back_call_price,
                "Net Cost": net_cost,
                "Future VIX (Day+7)": future_price,
                "Remaining Value of Long Call": remaining_call_value,
                "Profit": profit
            })

# Convert to DataFrame
calendar_df = pd.DataFrame(calendar_spread_results)

# Create a result DataFrame
put_simulation_df = pd.DataFrame(put_simulation_results)

if not calendar_df.empty and not put_simulation_df.empty:
    put_returns = put_simulation_df["Total Return (Simulated)"]
    put_trading_days = put_simulation_df["Spike Duration (Days)"]
    average_daily_returns = []

    # Loop through each trade
    for i in range(len(put_returns)):
        if put_trading_days[i] != 0:  # prevent division by zero
            avg_return = put_returns[i] / put_trading_days[i]
        else:
            avg_return = 0
        for j in range(put_trading_days[i]):
          average_daily_returns.append(float(avg_return))
    calendar_returns = list(calendar_df["Profit"].apply(float))
    strategy_returns = average_daily_returns + calendar_returns
    sharpe_ratio = np.mean(strategy_returns)/np.std(strategy_returns)
    print(sum(strategy_returns))
    print(sum(strategy_returns)/sum(total_cost))
    print(len(strategy_returns))
    wins = sum(1 for pnl in strategy_returns if pnl > 0)
    print(wins/len(strategy_returns))
    print(sharpe_ratio)

# # Extract total returns for the Sharpe ratio calculation
# returns = [result['Total Return (Simulated)'] for result in put_simulation_results]
# trades = [result['Spike Duration (Days)'] for result in put_simulation_results]

# # Calculate mean and standard deviation of returns
# mean_return = np.mean(returns)
# std_dev_return = np.std(returns)

# # Assuming a risk-free rate of X%
# risk_free_rate = X

# # Calculate Sharpe Ratio
# sharpe_ratio = (mean_return - risk_free_rate) / std_dev_return
# # print(sum(trades), sum(returns), sharpe_ratio)

# Volatility of Volatility

In [ ]:
import pandas as pd

# Load and prepare VIX data
vix_df = pd.read_csv("CBOE Volatility Index Historical Data 2021 to 2025.csv")
vix_df["Date"] = pd.to_datetime(vix_df["Date"])
vix_df["Price"] = pd.to_numeric(vix_df["Price"], errors="coerce")
vix_df = vix_df.sort_values("Date").reset_index(drop=True)

# Calculate daily returns of VIX
vix_df["VIX_Return"] = vix_df["Price"].pct_change()

# Calculate the rolling X-day standard deviation (VoV) of daily VIX returns
vix_df["VoV_10d"] = vix_df["VIX_Return"].rolling(window=10).std()

# Function to calculate VoV for a given date
def calculate_vov(input_date_str):
    # Convert input date string to datetime
    input_date = pd.to_datetime(input_date_str)

    # Make sure the date exists in the dataset
    if input_date not in vix_df["Date"].values:
        return f"Date {input_date} not found in data"

    # Get the VoV for the specified date
    vov_on_date = vix_df.loc[vix_df["Date"] == input_date, "VoV_10d"].values[0]

    if pd.isna(vov_on_date):
        return f"VoV data is missing for {input_date}"

    return f"VoV for {input_date.strftime('%Y-%m-%d')}: {vov_on_date:.4f}"

# Example usage:
input_date = "2025-06-13"  # Provide the date as 'YYYY-MM-DD'
vov_result = calculate_vov(input_date)
print(vov_result)


# Shitty buy put

In [ ]:
# Get a set of all indices that are part of any spike group
spike_indices = set(idx for group in spike_groups for idx in group)

# Loop through all indices in the DataFrame that are NOT in any spike group
non_spike_indices = [i for i in range(len(vix_df)) if i not in spike_indices]

# Loop through these non-spike days for custom strategy
# current_contracts elements [strike price, cost]
current_contract = []
trade_dates = []
trades = []
cost = []

# Instead of long put do sell call spreads
for i in non_spike_indices:
    current_date = vix_df.loc[i, "Date"]
    current_price = vix_df.loc[i, "Price"]
    current_volatility = vix_df.loc[i, "VoV_10d"] * VOV_SCALE
    strike_price = np.floor(current_price)
    # print(current_date.weekday())

    if pd.isna(current_volatility):
        continue
    # if not current_contract:
    #     long_put_price = black_scholes_put(S=current_price, K=strike_price, T=X/X, r=X, sigma=current_volatility)
    #     current_contract = [np.floor(current_price), long_put_price]
        # if int(current_date.weekday()) == X:
        #     current_contract = []
    if int(current_date.weekday()) == 2:
        if current_contract != []:
            trades.append(max(current_contract[0] - current_price, 0) - current_contract[1])
            trade_dates.append(current_date)
            cost.append(current_contract[1])
        long_put_price = black_scholes_put(S=current_price, K=strike_price, T=EXPIRY_WEEKS / 52, r=0.01, sigma=current_volatility)
        # short_put_price = black_scholes_put(S=current_price, K=np.floor(current_price) - X, T=X/X, r=X, sigma=current_volatility)
        # current_contract = [np.floor(current_price), long_put_price, np.floor(current_price) - X, short_put_price]
        current_contract = [strike_price, long_put_price]
        # current_contract = []
    elif current_contract != []:
        if current_price <= current_contract[0] - 1:
            days_to_expire = 1
            if current_date.weekday() < 2:
                days_to_expire = 2 - current_date.weekday()
            else:
                days_to_expire = 3 + current_date.weekday()
            long_put_price = black_scholes_put(S=current_price, K=current_contract[0], T=days_to_expire/365, r=0.01, sigma=current_volatility)
            # short_put_price = black_scholes_put(S=current_price, K=current_contract[X] , T=days_to_expire/X, r=X, sigma=current_volatility)
            # trades.append(long_put_price - current_contract[X] + current_contract[X] - short_put_price)
            trades.append(long_put_price - current_contract[1])
            trade_dates.append(current_date)
            cost.append(current_contract[1])
            # new_long_put_price = black_scholes_put(S=current_price, K=strike_price, T=X/X, r=X, sigma=current_volatility)
            # new_short_put_price = black_scholes_put(S=current_price, K=np.floor(current_price) - X, T=X/X, r=X, sigma=current_volatility)
            # current_contract = [np.floor(current_price), new_long_put_price, np.floor(current_price) - X, new_short_put_price]
            # current_contract = [strike_price, new_long_put_price]
            current_contract = []
            # if int(current_date.weekday()) == X:
            #     current_contract = []
trading_return = sum(trades)
trade_df = pd.DataFrame({
    "Date": trade_dates,
    "Trade Return": trades
})

# Full backtest with stop profit and shit

In [ ]:
# Simulate the returns from buying X SPY call every day during the spike , roll
put_simulation_results = []

for group in spike_groups:
    start_idx = group[0]
    end_idx = group[-1]
    current_contracts = []

    # Strike prices: VIX closing prices for each day of the spike
    strike_prices = np.floor(vix_df.loc[group, "Price"].tolist())

    # Assume a constant risk-free rate (r) and expiration time (T)
    r = 0.001  # Risk-free rate
    T = 1 / 52  # X weeks to expiration in years

    # Calculate the spike end threshold: average of the highest VIX price and pre-spike X-day average
    spike_prices = vix_df.loc[group, "Price"].tolist()
    spike_high = max(spike_prices)
    pre_spike_avg = vix_df.loc[start_idx, "VIX_5d_avg"]

    if pd.isna(pre_spike_avg):
        continue  # Skip if not enough data for pre-spike avg

    # Spike end threshold = average of pre-spike X-day avg and spike high
    spike_end_threshold = END_W * spike_high + (1 - END_W) * pre_spike_avg

    # Simulate buying X put each day during the spike and settling at post-spike expiration
    total_return = 0
    for i, strike_price in enumerate(strike_prices):
        # Use VoV as implied volatility for Black-Scholes model
        implied_volatility = vix_df.loc[group[i], "VIXW_Call_IV_5d"]
        strike_price = strike_price

        # Calculate the Black-Scholes put price for the current day
        put_price = black_scholes_put(S=vix_df.loc[group[i], "Price"], K=strike_price, T=T, r=r, sigma=implied_volatility)

        # Get the expiration date (X days from the buy date)
        purchase_date = vix_df.loc[group[i], "Date"]
        expiration_date = get_expiration_date(purchase_date)
        days_to_expire = 14
        long_put_price = 0

        current_contracts.append([strike_price, put_price, expiration_date])

    trades = []
    if end_idx + 10 < len(vix_df):
        for i in range(end_idx, end_idx + 10):
            current_date = vix_df.loc[i, "Date"]
            current_price = vix_df.loc[i, "Price"]
            current_volatility = vix_df.loc[i, "VIXW_Call_IV_5d"]
            for j in range(len(current_contracts)):
                current_contract = current_contracts[j]
                if not current_contract:
                    continue
                if days_to_expire == 0:
                    profit = max((current_contract[0] - current_price), 0) - current_contract[1]
                    trades.append(profit/current_contract[1])
                    # print(profit, current_contract[X])
                    # roi.append(profit/current_contract[X])

                    current_contracts[j] = []
                elif long_put_price > TAKE_PROFIT_MULT * current_contract[1]:
                    profit = long_put_price - current_contract[1]
                    trades.append(min(profit/current_contract[1], PROFIT_CAP))
                    # print(profit, current_contract[X])
                    current_contracts[j] = []
                days_to_expire = (current_contract[2] - current_date).days
                long_put_price = black_scholes_put(S=current_price, K=current_contract[0], T=days_to_expire/365, r=0.01, sigma=current_volatility)
    total_return = sum(trades)
    # print(total_return)

    # Store results for each spike
    put_simulation_results.append({
        "Start Date": vix_df.loc[start_idx, "Date"],
        "End Date": expiration_date,
        "Strike Prices": strike_prices,
        "Contract Prices" : put_price,
        "Total Return (Simulated)": total_return,
        "Spike Duration (Days)": len(group)
    })

# # Create a result DataFrame
put_simulation_df = pd.DataFrame(put_simulation_results)
# print(put_simulation_df)

spike_trading_return = list(put_simulation_df["Total Return (Simulated)"])
# print(spike_trading_return)
# print(spike_trading_return)
# sharpe_ratio = np.mean(trading_return)/np.std(trading_return)
# print(sharpe_ratio)